PREPARING THE TENSORS!

In [21]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scaling for x
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Encoding for labels
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Convert all to tensors
'''
Note: X,y are already NumpyArrays.
'''
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)
'''
X_train_tensor = torch.from_numpy(X_train).to(torch.float32)
X_test_tensor = torch.from_numpy(X_test).to(torch.float32)
y_train_tensor = torch.from_numpy(y_train).to(torch.float32)
y_test_tensor = torch.from_numpy(y_test).to(torch.float32)
'''

'\nX_train_tensor = torch.from_numpy(X_train).to(torch.float32)\nX_test_tensor = torch.from_numpy(X_test).to(torch.float32)\ny_train_tensor = torch.from_numpy(y_train).to(torch.float32)\ny_test_tensor = torch.from_numpy(y_test).to(torch.float32)\n'

BUILD MODEL ARCHITECTURE!

In [8]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

INITIALIZE THE MODEL, LOSS AND OPTIMISER!

In [9]:
lr = 1e-4
# initialize the built model
model = MyNN(len(X_train_tensor[0]))
# initialize loss from nn
loss_fun = nn.BCELoss()
# initialize the optimiser form torch
optim = torch.optim.SGD(model.parameters(), lr)

KNOWING THE IMPORTANCE OF DATALOADER!  
MINI-BATCH GRADIENT DESCENT: MANUAL IMPLEMENTATION!

In [10]:
# params
batch_size = 32
epochs = 100
num_samples = 100

for epoch in range(50):
    for st_idx in range(0, num_samples):
        # make batches
        end_idx = st_idx+batch_size
        X_batch = X_train_tensor[st_idx:end_idx]
        y_batch = y_train_tensor[st_idx:end_idx]
        # run the epoch
        optim.zero_grad()
        y_pred_tensor = model.forward(X_batch)
        loss = loss_fun(y_pred_tensor, y_batch.reshape(-1,1))
        loss.backward()
        optim.step()

    print(f"epoch: {epoch} with loss: {loss}")

epoch: 0 with loss: 0.6097492575645447
epoch: 1 with loss: 0.5939346551895142
epoch: 2 with loss: 0.578986406326294
epoch: 3 with loss: 0.5648435354232788
epoch: 4 with loss: 0.5514491200447083
epoch: 5 with loss: 0.5387504696846008
epoch: 6 with loss: 0.5266992449760437
epoch: 7 with loss: 0.515250563621521
epoch: 8 with loss: 0.504362940788269
epoch: 9 with loss: 0.4939986765384674
epoch: 10 with loss: 0.484122633934021
epoch: 11 with loss: 0.4747026860713959
epoch: 12 with loss: 0.46570906043052673
epoch: 13 with loss: 0.45711463689804077
epoch: 14 with loss: 0.4488939642906189
epoch: 15 with loss: 0.4410240054130554
epoch: 16 with loss: 0.4334832727909088
epoch: 17 with loss: 0.4262518286705017
epoch: 18 with loss: 0.4193114638328552
epoch: 19 with loss: 0.4126451909542084
epoch: 20 with loss: 0.4062372148036957
epoch: 21 with loss: 0.4000730514526367
epoch: 22 with loss: 0.3941391110420227
epoch: 23 with loss: 0.3884228467941284
epoch: 24 with loss: 0.3829125165939331
epoch: 25 wi

DATASET CLASS!  
MINI-BATCH GRADIENT DESCENT: MANUAL IMPLEMENTATION!


In [11]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor

    def __len__(self):
        return len(self.X_train_tensor)

    def __getitem__(self, index):
        return self.X_train_tensor[index], self.y_train_tensor[0]


In [12]:
# Sample Usage!
dataset = CustomDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [13]:
# Loading the CustomDataset!
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [14]:
# Loading the DataLoader!
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [15]:
for epoch in range(epochs):
    for x_train_batch, y_train_batch in train_loader:
        optim.zero_grad()
        y_pred_batch = model.forward(x_train_batch)
        loss = loss_fun(y_pred_batch, y_train_batch.reshape(-1,1))
        loss.backward()
        optim.step()
    print(f"Epoch: {epoch}, Loss: {loss.item()}")

Epoch: 0, Loss: 1.158063292503357
Epoch: 1, Loss: 1.155417561531067
Epoch: 2, Loss: 1.1527798175811768
Epoch: 3, Loss: 1.150149941444397
Epoch: 4, Loss: 1.1475276947021484
Epoch: 5, Loss: 1.1449135541915894
Epoch: 6, Loss: 1.1423072814941406
Epoch: 7, Loss: 1.1397087574005127
Epoch: 8, Loss: 1.1371181011199951
Epoch: 9, Loss: 1.1345351934432983
Epoch: 10, Loss: 1.1319602727890015
Epoch: 11, Loss: 1.1293933391571045
Epoch: 12, Loss: 1.1268341541290283
Epoch: 13, Loss: 1.1242830753326416
Epoch: 14, Loss: 1.1217397451400757
Epoch: 15, Loss: 1.1192042827606201
Epoch: 16, Loss: 1.116676926612854
Epoch: 17, Loss: 1.1141570806503296
Epoch: 18, Loss: 1.1116453409194946
Epoch: 19, Loss: 1.1091415882110596
Epoch: 20, Loss: 1.1066455841064453
Epoch: 21, Loss: 1.1041576862335205
Epoch: 22, Loss: 1.1016775369644165
Epoch: 23, Loss: 1.0992052555084229
Epoch: 24, Loss: 1.096740961074829
Epoch: 25, Loss: 1.0942846536636353
Epoch: 26, Loss: 1.0918362140655518
Epoch: 27, Loss: 1.089395523071289
Epoch: 2

MODEL EVALUATION - BATCH WISE

In [16]:
with torch.no_grad():
    for i, (X_test_batch, y_test_batch) in enumerate(test_loader):
        y_pred_batch = model.forward(X_test_batch)

        y_pred_batch = (y_pred_batch>0.9).float()
        accuracy = (y_pred_batch==y_test_batch).float().mean()

        print(f"Accuracy of Batch {i}: {accuracy}")


Accuracy of Batch 0: 1.0
Accuracy of Batch 1: 0.9375
Accuracy of Batch 2: 0.96875
Accuracy of Batch 3: 1.0


MODEL EVALUATION - OVERALL WISE

In [17]:
with torch.no_grad():
    correct = 0
    total = 0
    for X_test_batch, y_test_batch in test_loader:
        y_pred_batch = model(X_test_batch)
        y_pred_batch = (y_pred_batch > 0.9).float()
        correct += (y_pred_batch == y_test_batch.reshape(-1, 1)).sum().item()
        total += y_test_batch.size(0)
    accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9737
